In [2]:
import numpy as np
from scipy.spatial import cKDTree

In [3]:
data_path = "../DIMON_training_data_healthy.npz"
dataset = np.load(data_path)

theta = dataset['theta']   
pacing = dataset['pacing'] 
u_all = dataset['u_data']  
cobiveco = dataset['cobiveco']
anisotropy = dataset['ref_anisotropy'] 
cartesian_coords = dataset['cartesian_coords'] 

In [4]:
u_all.shape

(125, 9, 50797)

In [5]:
def calculate_discrete_gradient(x_coords, u_data, k=15):
    tree = cKDTree(x_coords)
    # Find k nearest neighbors for all points
    dist, idx = tree.query(x_coords, k=k)
    
    num_nodes = x_coords.shape[0]
    grads = np.zeros((num_nodes, 3))

    for i in range(num_nodes):
        # Local neighborhood
        neighbors = idx[i]
        
        # Relative coordinates (A matrix)
        A = x_coords[neighbors[1:]] - x_coords[i] 
        # Activation differences (b vector)
        b = u_data[neighbors[1:]] - u_data[i]
        
        # Solve Least Squares: g = inv(A.T @ A) @ A.T @ b
        g, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        grads[i] = g
        
    return grads

In [10]:
u_11 = u_all[0,0,:]
grad_u= calculate_discrete_gradient(cartesian_coords,u_11)

In [8]:
# 1. Extract unit vectors
f = anisotropy[:, 0:3] # (N, 3)
s = anisotropy[:, 3:6] # (N, 3)
n = anisotropy[:, 6:9] # (N, 3)

# 2. Define velocities
vf, vs, vn = 640.0, 240.0, 240.0

# 3. Build the (N, 3, 3) tensor field using batch outer products
# 'ni,nj->nij' computes the outer product for each row n
D = (vf**2) * np.einsum('ni,nj->nij', f, f) + \
    (vs**2) * np.einsum('ni,nj->nij', s, s) + \
    (vn**2) * np.einsum('ni,nj->nij', n, n)

In [14]:
# grad_u is (N, 3)
# D is (N, 3, 3)

# 'ni,nij,nj->n' performs the quadratic form: grad_u^T @ D @ grad_u for each node
inner_product = np.einsum('ni,nij,nj->n', grad_u, D, grad_u)

# Compute the residual
lhs = np.sqrt(np.maximum(inner_product, 1e-9))
residual = lhs - 1.0 # This is (N,)
alt_residual = np.maximum(inner_product, 1e-9) -1

In [15]:
alt_residual.mean()

np.float64(0.5600841643028185)

In [16]:
def calculate_all_residuals(u_all, cartesian_coords, anisotropy, vf=640.0, vs=240.0, vn=240.0):
    num_hearts, num_sims, num_nodes = u_all.shape
    
    # 1. Pre-calculate Diffusion Tensor D (N, 3, 3)
    f, s, n = anisotropy[:, 0:3], anisotropy[:, 3:6], anisotropy[:, 6:9]
    D = (vf**2) * np.einsum('ni,nj->nij', f, f) + \
        (vs**2) * np.einsum('ni,nj->nij', s, s) + \
        (vn**2) * np.einsum('ni,nj->nij', n, n)

    # 2. Setup Neighborhood for Gradients
    k = 15
    tree = cKDTree(cartesian_coords)
    dist, idx = tree.query(cartesian_coords, k=k)
    
    # Pre-calculate the A matrices and their pseudo-inverses for all nodes
    # This significantly speeds up the loop over 125x9 simulations
    A_stack = cartesian_coords[idx[:, 1:]] - cartesian_coords[:, np.newaxis, :] # (N, k-1, 3)
    # Pseudo-inverse: (A^T A)^-1 A^T
    # We use a loop for the pinv as it's node-specific
    A_pinv = np.zeros((num_nodes, 3, k-1))
    for i in range(num_nodes):
        A_pinv[i] = np.linalg.pinv(A_stack[i])

    # 3. Initialize Results Array
    mean_residuals = np.zeros((num_hearts, num_sims))

    print(f"Processing {num_hearts} hearts...")
    for h in range(num_hearts):
        for s_idx in range(num_sims):
            u_current = u_all[h, s_idx, :]
            
            # Discrete Gradient Calculation via pre-calculated Pseudo-inverse
            # Delta u for all neighbors: (N, k-1)
            b = u_current[idx[:, 1:]] - u_current[:, np.newaxis]
            
            # Solve grad = A_pinv @ b -> (N, 3, k-1) @ (N, k-1, 1)
            grad_u = np.einsum('nij,nj->ni', A_pinv, b)
            
            # Physics Check (Eikonal LHS)
            inner_product = np.einsum('ni,nij,nj->n', grad_u, D, grad_u)
            lhs = np.sqrt(np.maximum(inner_product, 1e-9))
            
            # Calculate Mean Absolute Residual for this specific simulation
            mean_residuals[h, s_idx] = np.mean(np.abs(lhs - 1.0))
            
        if (h + 1) % 10 == 0:
            print(f"Heart {h+1}/{num_hearts} complete.")

    return mean_residuals

# --- Execution ---
residual_matrix = calculate_all_residuals(u_all, cartesian_coords, anisotropy)

print("\nFinal Residual Matrix Shape:", residual_matrix.shape)
print("Global Mean Residual:", np.mean(residual_matrix))

Processing 125 hearts...
Heart 10/125 complete.
Heart 20/125 complete.
Heart 30/125 complete.
Heart 40/125 complete.
Heart 50/125 complete.
Heart 60/125 complete.
Heart 70/125 complete.
Heart 80/125 complete.
Heart 90/125 complete.
Heart 100/125 complete.
Heart 110/125 complete.
Heart 120/125 complete.

Final Residual Matrix Shape: (125, 9)
Global Mean Residual: 0.432047102580759


In [20]:
residual_matrix.mean(), residual_matrix.std()

(np.float64(0.432047102580759), np.float64(0.06363853300483825))